# Kaikōura: U-Net, Random Forest and GNS road comparison

Run in Colab to compare the saved U-Net (epoch 77, threshold 0.31), saved Random Forest (150 trees, threshold 0.28), and GNS V3 mapped landslide reference on the same road-selected imagery. No new training or threshold tuning is performed.

The original U-Net road result appears first. The final comparison includes post-event RGB, GNS labels, both prediction masks, error maps, shared-pixel metrics, and three road-overlap layers. GNS is a landslide inventory, **not confirmed road-blockage truth**. The road-selected tiles may overlap training/validation: these are integration reference scores, not a new independent test.

Use your own registered Earth Engine compute project. The final reference comparison requires read access to the shared GNS assets; the saved RF is loaded from the shared Google Drive artifact. A copy is also provided as `rf_150_trees.joblib.gz` in this repository; if needed, upload it to `/content/kaikoura_unet_roads/rf_150_trees.joblib.gz` before running the RF loader. The U-Net weights are included in the notebook. No model weights need to be pasted into a cell manually.

Unobserved road sections remain **unknown**. A predicted intersection indicates a potential impact; the 10 m imagery does not establish full blockage, remaining vehicle clearance, or safe passage. The interactive Esri basemap is illustrative and is not verified 2016 imagery.


In [ ]:
%pip install -q earthengine-api geopandas rasterio folium requests pyproj matplotlib scikit-learn==1.6.1


In [ ]:
from pathlib import Path
import json, hashlib, math, time
import ee

EE_PROJECT = 'x-band-508903' # @param {type:"string"}
ROOT = Path('/content/kaikoura_unet_roads')
ROOT.mkdir(exist_ok=True)
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate(auth_mode='colab')
    ee.Initialize(project=EE_PROJECT)
assert ee.Number(1).getInfo() == 1
print('Earth Engine connected:', EE_PROJECT)


## Load the saved model

The included checkpoint contains model weights and metadata only. It is the existing cloud-trained model, not a new training run. The 0.31 threshold was selected on the earlier validation set and is frozen here. The previous F1/IoU results are landslide pixel metrics, not road-blockage accuracy.

In [ ]:
# @title Load saved U-Net (weights included; no edits needed)
import base64, gzip
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset

FEATURE_NAMES = ["B2", "B3", "B4", "B8", "B11", "B12", "elevation", "slope"]


def normalize_features(arr):
    """Same fixed, label-independent transform for both models."""
    x = np.nan_to_num(arr[:8].astype("float32"), nan=0, posinf=0, neginf=0)
    x[:6] = np.clip(x[:6], 0, 1)
    x[6] = np.clip(x[6], 0, 3000) / 3000.0
    x[7] = np.clip(x[7], 0, 90) / 90.0
    return x


def read_tile(record):
    with rasterio.open(record["path"]) as ds:
        arr = ds.read()
    return normalize_features(arr), (arr[8] > 0.5), (arr[9] > 0.5)


class TileDataset(Dataset):
    def __init__(self, records, augment=False):
        self.records, self.augment = records, augment

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        x, y, valid = read_tile(self.records[i])
        y, valid = y[None].astype("float32"), valid[None].astype("float32")
        if self.augment:
            if random.random() < 0.5:
                x, y, valid = x[:, :, ::-1], y[:, :, ::-1], valid[:, :, ::-1]
            if random.random() < 0.5:
                x, y, valid = x[:, ::-1, :], y[:, ::-1, :], valid[:, ::-1, :]
            k = random.randrange(4)
            x, y, valid = [np.rot90(a, k, axes=(1, 2)) for a in (x, y, valid)]
        return tuple(torch.from_numpy(a.copy()) for a in (x, y, valid))


class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.GroupNorm(4, out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.GroupNorm(4, out_channels), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class PostEventUNet(nn.Module):
    def __init__(self, in_channels=8, base=8):
        super().__init__()
        self.enc = nn.ModuleList([
            ConvBlock(in_channels, base), ConvBlock(base, base * 2),
            ConvBlock(base * 2, base * 4), ConvBlock(base * 4, base * 8),
        ])
        self.pool = nn.MaxPool2d(2)
        self.dec = nn.ModuleList([
            ConvBlock(base * 8 + base * 4, base * 4),
            ConvBlock(base * 4 + base * 2, base * 2),
            ConvBlock(base * 2 + base, base),
        ])
        self.head = nn.Conv2d(base, 1, 1)

    def forward(self, x):
        skips = []
        for i, block in enumerate(self.enc):
            x = block(self.pool(x) if i else x)
            skips.append(x)
        for block, skip in zip(self.dec, reversed(skips[:-1])):
            x = nn.functional.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            x = block(torch.cat([x, skip], dim=1))
        return self.head(x)



MODEL_METADATA = {'checkpoint_sha256': 'f055f9862dfe911a663b0310266fb9c4bca54843457f91b96b4ce32c6f986c70', 'original_checkpoint_sha256': 'c6c15549ae05f698b57d759b94419cb3f4fad263da19b9ab0e8a0926e65c81aa', 'epoch': 77, 'positive_weight': 1, 'threshold': 0.31, 'features': ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'elevation', 'slope'], 'base': 8, 'grid': [10, 0, 1641080, 0, -10, 5350110], 'aoi': [173.5, -42.5, 174, -42.0]}
# Embedded saved weights keep this notebook independent of an expired runtime.
_MODEL_PAYLOAD = '''
...
'''
checkpoint_path = ROOT / 'unet_w1_epoch77_inference.pt'
raw = gzip.decompress(base64.b64decode(_MODEL_PAYLOAD))
assert hashlib.sha256(raw).hexdigest() == MODEL_METADATA['checkpoint_sha256']
checkpoint_path.write_bytes(raw)
del _MODEL_PAYLOAD, raw
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(min(4, torch.get_num_threads()))
checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
assert checkpoint['epoch'] == 77
assert checkpoint['config']['base'] == 8
THRESHOLD = float(checkpoint['metrics']['val_threshold'])
assert abs(THRESHOLD - 0.31) < 1e-8
model = PostEventUNet(in_channels=8, base=8).to(DEVICE)
model.load_state_dict(checkpoint['state_dict'], strict=True)
model.eval()
print('Loaded U-Net: epoch 77 | positive weight 1 | threshold', THRESHOLD, '| device', DEVICE)
print('Features:', FEATURE_NAMES)


In [ ]:
# @title Load road-analysis tools (no edits needed)
# Embedded reviewed integration helpers; no external repository code is executed.
import sys
MODULE_SOURCES = {'road_impact.py': '"""Component 2: road-centreline overlap, not confirmed blockage or passability."""\nfrom pathlib import Path\nimport argparse, json\nimport geopandas as gpd\nimport pandas as pd\nfrom shapely import make_valid, union_all\nfrom shapely.geometry import box, LineString\nfrom shapely.ops import linemerge\n\nAOI = (173.5, -42.5, 174.0, -42.0)\nCRS = \'EPSG:2193\'\nVEHICLE_ROADS = {\'motorway\',\'motorway_link\',\'trunk\',\'trunk_link\',\'primary\',\'primary_link\',\n \'secondary\',\'secondary_link\',\'tertiary\',\'tertiary_link\',\'unclassified\',\'residential\',\n \'living_street\',\'service\',\'track\',\'road\'}\n\ndef line_parts(geom):\n    if geom.is_empty: return []\n    if geom.geom_type == \'LineString\': return [geom] if geom.length > 0 else []\n    if hasattr(geom, \'geoms\'):\n        return [p for child in geom.geoms for p in line_parts(child)]\n    return []\n\ndef lengths(line, hazard):\n    pieces = line_parts(line.intersection(hazard))\n    if not pieces: return 0.0, 0.0, []\n    merged = union_all(pieces)\n    if merged.geom_type == \'MultiLineString\': merged = linemerge(merged)\n    runs = line_parts(merged)\n    return sum(p.length for p in runs), max(p.length for p in runs), runs\n\ndef read_hazards(path, roi):\n    data = gpd.read_file(path)\n    if data.crs is None: raise ValueError(f\'Missing CRS: {path}\')\n    data = data.to_crs(CRS)\n    data = data.loc[data.geometry.intersects(roi)].copy()\n    data.geometry = data.geometry.map(make_valid).intersection(roi)\n    return data\n\ndef run(roads_path, source_path, debris_path, out_dir):\n    out = Path(out_dir);out.mkdir(parents=True,exist_ok=True)\n    roi = gpd.GeoSeries([box(*AOI)],crs=4326).to_crs(CRS).iloc[0]\n    roads = gpd.read_file(roads_path)\n    if roads.crs is None: raise ValueError(\'Road input has no CRS\')\n    if not roads.geom_type.isin([\'LineString\',\'MultiLineString\']).all():\n        raise ValueError(\'Road input must contain lines: Kaikoura.shp is an administrative polygon. Use kai_route.gpkg.\')\n    if \'full_id\' not in roads or roads.full_id.isna().any() or roads.full_id.duplicated().any():\n        raise ValueError(\'Expected unique full_id from agreed team road file.\')\n    raw_count = len(roads)\n    # Exclude railways, buildings and pedestrian-only paths. Access restrictions retained as attributes.\n    roads = roads.loc[roads.highway.isin(VEHICLE_ROADS)].to_crs(CRS).copy()\n    roads = roads.loc[roads.geometry.intersects(roi)].copy()\n    roads.geometry = roads.geometry.intersection(roi)\n    rows=[]\n    for _,row in roads.iterrows():\n        for i,part in enumerate(line_parts(row.geometry)):\n            attrs={k:row.get(k) for k in [\'full_id\',\'osm_id\',\'name\',\'highway\',\'surface\',\'access\',\'motor_vehicle\',\'bridge\',\'tunnel\',\'width\']}\n            rows.append({**attrs,\'road_id\':f\'{row.full_id}:part{i}\', \'part_index\':i,\'geometry\':part})\n    if not rows: raise ValueError(\'No road lines in AOI\')\n    segments = gpd.GeoDataFrame(rows,crs=CRS)\n    segments[\'road_length_m\'] = segments.length\n    source = read_hazards(source_path,roi);debris = read_hazards(debris_path,roi)\n    source_union = union_all(source.geometry.values)\n    debris_union = union_all(debris.geometry.values)\n    hazard = union_all([source_union,debris_union])  # Prevent double counting.\n    source_index = source.sindex\n    debris_index = debris.sindex\n    results=[];affected_rows=[]\n    for row in segments.itertuples():\n        near = row.geometry.buffer(30)\n        local_source = union_all(source.geometry.iloc[source_index.query(near, predicate=\'intersects\')].values)\n        local_debris = union_all(debris.geometry.iloc[debris_index.query(near, predicate=\'intersects\')].values)\n        local_hazard = union_all([local_source,local_debris])\n        total, longest, pieces = lengths(row.geometry,local_hazard)\n        record={\'road_id\':row.road_id,\'affected_length_m\':total,\n            \'max_affected_run_m\':longest,\'affected_fraction\':total/row.road_length_m,\n            \'source_overlap_m\':lengths(row.geometry,local_source)[0],\n            \'debris_overlap_m\':lengths(row.geometry,local_debris)[0],\n            \'impact_status\':\'potentially_affected\' if total>0 else \'no_mapped_overlap\',\n            \'hazard_source\':\'GNS V3 mapped landslide inventory\'}\n        # Sensitivity to positional error, not measured road width or proven damage.\n        for distance in [10,30]:\n            record[f\'near_hazard_{distance}m_overlap_m\']=lengths(row.geometry,local_hazard.buffer(distance))[0]\n        results.append(record)\n        for i,piece in enumerate(pieces):\n            affected_rows.append({\'road_id\':row.road_id,\'run_id\':i,\'length_m\':piece.length,\'geometry\':piece})\n    result = segments.merge(pd.DataFrame(results),on=\'road_id\',validate=\'one_to_one\')\n    assert (result.affected_length_m <= result.road_length_m + 1e-6).all()\n    assert result.road_id.is_unique\n    result.drop(columns=\'geometry\').to_csv(out/\'road_impact.csv\',index=False)\n    result.to_file(out/\'road_impact.gpkg\',layer=\'road_segments\',driver=\'GPKG\')\n    result.to_crs(4326).to_file(out/\'road_impact.geojson\',driver=\'GeoJSON\')\n    if affected_rows:\n        gpd.GeoDataFrame(affected_rows,crs=CRS).to_file(out/\'affected_runs.gpkg\',driver=\'GPKG\')\n    import folium\n    m=folium.Map(location=[-42.25,173.75],zoom_start=10,tiles=\'OpenStreetMap\')\n    folium.GeoJson(gpd.GeoSeries([hazard],crs=CRS).to_crs(4326).__geo_interface__,\n        name=\'GNS mapped landslides\',style_function=lambda f:{\'color\':\'orange\',\'weight\':1,\'fillOpacity\':0.25}).add_to(m)\n    folium.GeoJson(result.to_crs(4326).to_json(),name=\'Road segments\',\n        style_function=lambda f:{\'color\':\'red\' if f[\'properties\'][\'affected_length_m\']>0 else \'blue\',\'weight\':3},\n        tooltip=folium.GeoJsonTooltip(fields=[\'road_id\',\'name\',\'affected_length_m\',\'max_affected_run_m\',\'impact_status\'])).add_to(m)\n    folium.LayerControl().add_to(m);m.save(out/\'road_impact_map.html\')\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    fig,ax=plt.subplots(figsize=(9,9))\n    gpd.GeoSeries([hazard],crs=CRS).plot(ax=ax,color=\'#e8c685\',alpha=0.6)\n    result.plot(ax=ax,color=\'#64748b\',linewidth=0.8)\n    if affected_rows: gpd.GeoDataFrame(affected_rows,crs=CRS).plot(ax=ax,color=\'red\',linewidth=2)\n    ax.set_title(\'Kaikōura: mapped landslides and potential road impacts\\nRed = centreline overlap; not confirmed road closure\')\n    ax.set_xlabel(\'NZTM easting (m)\');ax.set_ylabel(\'NZTM northing (m)\')\n    fig.tight_layout();fig.savefig(out/\'road_impact_map.png\',dpi=180);plt.close(fig)\n    summary={\'input_features\':raw_count,\'road_parts_in_aoi\':len(result),\n        \'potentially_affected_parts\':int((result.affected_length_m>0).sum()),\n        \'total_overlap_m\':float(result.affected_length_m.sum()),\n        \'max_continuous_overlap_m\':float(result.max_affected_run_m.max()),\n        \'aoi_wgs84\':AOI,\'metric_crs\':CRS,\'source_polygons_in_aoi\':len(source),\'debris_polygons_in_aoi\':len(debris),\n        \'limitations\':[\'OSM snapshot date unknown; may reflect post-2016 road realignment.\',\n        \'Road file is a subset, not verified complete regional routable graph.\',\n        \'No overlap is not evidence of safe passage. Overlap is not confirmed closure.\',\n        \'Lengths measured on road centrelines; no vehicle clearance or road width inference.\',\n        \'part IDs identify clipped line components, not routing graph edges split at junctions.\',\n        \'10/30 m buffers are positional sensitivity checks, not extra confirmed impacts.\']}\n    (out/\'summary.json\').write_text(json.dumps(summary,indent=2))\n    print(json.dumps(summary,indent=2));return result\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--roads\',required=True);p.add_argument(\'--source\',required=True)\n    p.add_argument(\'--debris\',required=True);p.add_argument(\'--out\',default=\'outputs/component2\')\n    args=p.parse_args();run(args.roads,args.source,args.debris,args.out)\n', 'landslide_benchmark.py': 'import numpy as np\nimport torch\nfrom torch import nn\nfrom torch.utils.data import Dataset\nFEATURE_NAMES = ["B2", "B3", "B4", "B8", "B11", "B12", "elevation", "slope"]\n\n\ndef normalize_features(arr):\n    """Same fixed, label-independent transform for both models."""\n    x = np.nan_to_num(arr[:8].astype("float32"), nan=0, posinf=0, neginf=0)\n    x[:6] = np.clip(x[:6], 0, 1)\n    x[6] = np.clip(x[6], 0, 3000) / 3000.0\n    x[7] = np.clip(x[7], 0, 90) / 90.0\n    return x\n\n\ndef read_tile(record):\n    with rasterio.open(record["path"]) as ds:\n        arr = ds.read()\n    return normalize_features(arr), (arr[8] > 0.5), (arr[9] > 0.5)\n\n\nclass TileDataset(Dataset):\n    def __init__(self, records, augment=False):\n        self.records, self.augment = records, augment\n\n    def __len__(self):\n        return len(self.records)\n\n    def __getitem__(self, i):\n        x, y, valid = read_tile(self.records[i])\n        y, valid = y[None].astype("float32"), valid[None].astype("float32")\n        if self.augment:\n            if random.random() < 0.5:\n                x, y, valid = x[:, :, ::-1], y[:, :, ::-1], valid[:, :, ::-1]\n            if random.random() < 0.5:\n                x, y, valid = x[:, ::-1, :], y[:, ::-1, :], valid[:, ::-1, :]\n            k = random.randrange(4)\n            x, y, valid = [np.rot90(a, k, axes=(1, 2)) for a in (x, y, valid)]\n        return tuple(torch.from_numpy(a.copy()) for a in (x, y, valid))\n\n\nclass ConvBlock(nn.Module):\n    def __init__(self, in_channels, out_channels):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Conv2d(in_channels, out_channels, 3, padding=1),\n            nn.GroupNorm(4, out_channels), nn.ReLU(inplace=True),\n            nn.Conv2d(out_channels, out_channels, 3, padding=1),\n            nn.GroupNorm(4, out_channels), nn.ReLU(inplace=True),\n        )\n\n    def forward(self, x):\n        return self.net(x)\n\n\nclass PostEventUNet(nn.Module):\n    def __init__(self, in_channels=8, base=8):\n        super().__init__()\n        self.enc = nn.ModuleList([\n            ConvBlock(in_channels, base), ConvBlock(base, base * 2),\n            ConvBlock(base * 2, base * 4), ConvBlock(base * 4, base * 8),\n        ])\n        self.pool = nn.MaxPool2d(2)\n        self.dec = nn.ModuleList([\n            ConvBlock(base * 8 + base * 4, base * 4),\n            ConvBlock(base * 4 + base * 2, base * 2),\n            ConvBlock(base * 2 + base, base),\n        ])\n        self.head = nn.Conv2d(base, 1, 1)\n\n    def forward(self, x):\n        skips = []\n        for i, block in enumerate(self.enc):\n            x = block(self.pool(x) if i else x)\n            skips.append(x)\n        for block, skip in zip(self.dec, reversed(skips[:-1])):\n            x = nn.functional.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)\n            x = block(torch.cat([x, skip], dim=1))\n        return self.head(x)\n\n', 'unet_road_integration.py': '"""Inference-only U-Net / road integration with explicit observation coverage.\n\nNo training is performed. Neural imports and forward passes occur only when\ninfer_road_tiles is explicitly called in Colab. Geometry utilities are independent.\n"""\nfrom pathlib import Path\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nimport hashlib\nimport json\nimport math\nimport time\n\nimport geopandas as gpd\nimport numpy as np\nimport pandas as pd\nimport rasterio\nfrom rasterio.transform import Affine\nfrom rasterio.features import shapes\nfrom rasterio.windows import Window\nimport requests\nfrom shapely import make_valid, union_all\nfrom shapely.geometry import box, shape, GeometryCollection\n\ntry:\n    from .road_impact import VEHICLE_ROADS, line_parts, lengths\nexcept ImportError:\n    from road_impact import VEHICLE_ROADS, line_parts, lengths\n\nAOI_WGS84 = (173.5, -42.5, 174.0, -42.0)\nMETRIC_CRS = "EPSG:2193"\nGRID_XMIN, GRID_YTOP = 1641080, 5350110\nSCALE, TILE_SIZE, BLOCK_TILES = 10, 128, 4\nBLOCK_SIZE = TILE_SIZE * BLOCK_TILES\nFEATURE_NAMES = ["B2", "B3", "B4", "B8", "B11", "B12", "elevation", "slope"]\nDEFAULT_THRESHOLD = 0.31\n\n\ndef aoi_metric():\n    return gpd.GeoSeries([box(*AOI_WGS84)], crs=4326).to_crs(METRIC_CRS).iloc[0]\n\n\ndef polygon_parts(geometry):\n    if geometry is None or geometry.is_empty:\n        return []\n    if geometry.geom_type == "Polygon":\n        return [geometry] if geometry.area > 0 else []\n    if hasattr(geometry, "geoms"):\n        return [part for child in geometry.geoms for part in polygon_parts(child)]\n    return []\n\n\ndef empty_geodataframe(columns, crs=METRIC_CRS):\n    data = {column: pd.Series(dtype="object") for column in columns if column != "geometry"}\n    return gpd.GeoDataFrame(data, geometry=gpd.GeoSeries([], crs=crs), crs=crs)\n\n\ndef _json_atomic(path, value):\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, indent=2))\n    temporary.replace(path)\n\n\ndef _file_hash(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef read_road_segments(road_path):\n    """Retain road attributes and IDs using the agreed component-2 ID convention."""\n    roads = gpd.read_file(road_path)\n    if roads.crs is None:\n        raise ValueError("Road input has no CRS.")\n    if not roads.geom_type.isin(["LineString", "MultiLineString"]).all():\n        raise ValueError("Use the kai_route.gpkg line layer; Kaikoura.shp is an administrative polygon.")\n    if "full_id" not in roads or roads.full_id.isna().any() or roads.full_id.duplicated().any():\n        raise ValueError("Road file needs unique non-null full_id values.")\n    if "highway" not in roads:\n        raise ValueError("Road file has no highway classification.")\n    raw_count = len(roads)\n    roads = roads.loc[roads.highway.isin(VEHICLE_ROADS)].to_crs(METRIC_CRS).copy()\n    roi = aoi_metric()\n    roads = roads.loc[roads.geometry.intersects(roi)].copy()\n    roads.geometry = roads.geometry.map(make_valid).intersection(roi)\n    rows = []\n    geometry_column = roads.geometry.name\n    for _, record in roads.iterrows():\n        attributes = {key: value for key, value in record.items() if key != geometry_column}\n        for index, part in enumerate(line_parts(record.geometry)):\n            rows.append(dict(attributes, road_id=f"{record.full_id}:part{index}",\n                             part_index=index, geometry=part))\n    if not rows:\n        raise ValueError("No vehicle-road line components intersect the agreed AOI.")\n    result = gpd.GeoDataFrame(rows, crs=METRIC_CRS)\n    result["road_length_m"] = result.length\n    if not result.road_id.is_unique:\n        raise ValueError("Exploded road IDs are not unique.")\n    result.attrs["input_feature_count"] = raw_count\n    result.attrs["road_file"] = str(road_path)\n    return result\n\n\ndef plan_road_tiles(segments):\n    """Select original 128-pixel tiles solely by clipped road geometry.\n\n    Include every intersected original tile, including mostly ocean/invalid tiles.\n    Group downloads in 4×4 blocks while retaining 128×128 inference windows.\n    """\n    if segments.crs is None:\n        raise ValueError("Road segments need a CRS.")\n    segments = segments.to_crs(METRIC_CRS)\n    roi = aoi_metric()\n    minx, miny, maxx, maxy = roi.bounds\n    nx = math.ceil((maxx - GRID_XMIN) / (TILE_SIZE * SCALE))\n    ny = math.ceil((GRID_YTOP - miny) / (TILE_SIZE * SCALE))\n    roads_union = union_all(segments.geometry.values).intersection(roi)\n    if roads_union.is_empty:\n        raise ValueError("No roads in AOI for inference planning.")\n    width = TILE_SIZE * SCALE\n    road_minx, road_miny, road_maxx, road_maxy = roads_union.bounds\n    cmin = max(0, math.floor((road_minx - GRID_XMIN) / width) - 1)\n    cmax = min(nx - 1, math.floor((road_maxx - GRID_XMIN) / width))\n    rmin = max(0, math.floor((GRID_YTOP - road_maxy) / width) - 1)\n    rmax = min(ny - 1, math.floor((GRID_YTOP - road_miny) / width))\n    tiles, blocks = [], {}\n    for row in range(rmin, rmax + 1):\n        for col in range(cmin, cmax + 1):\n            x, y = GRID_XMIN + col * width, GRID_YTOP - row * width\n            extent = box(x, y - width, x + width, y)\n            if not extent.intersects(roads_union):\n                continue\n            block_row, block_col = row // BLOCK_TILES, col // BLOCK_TILES\n            block_id = f"br{block_row:03d}_bc{block_col:03d}"\n            tile = {"tile": f"r{row:03d}_c{col:03d}", "row": row, "col": col,\n                    "block_id": block_id, "local_row": row % BLOCK_TILES,\n                    "local_col": col % BLOCK_TILES,\n                    "transform": [SCALE, 0, x, 0, -SCALE, y]}\n            tiles.append(tile)\n            if block_id not in blocks:\n                bx = GRID_XMIN + block_col * BLOCK_SIZE * SCALE\n                by = GRID_YTOP - block_row * BLOCK_SIZE * SCALE\n                blocks[block_id] = {"block_id": block_id, "block_row": block_row,\n                    "block_col": block_col, "transform": [SCALE, 0, bx, 0, -SCALE, by], "tiles": []}\n            blocks[block_id]["tiles"].append(tile["tile"])\n    return {"version": 1, "aoi_wgs84": list(AOI_WGS84), "crs": METRIC_CRS,\n        "grid_origin": [GRID_XMIN, GRID_YTOP], "scale": SCALE, "tile_size": TILE_SIZE,\n        "block_size": BLOCK_SIZE, "grid_columns": nx, "grid_rows": ny,\n        "selection": "all original-grid tiles intersecting agreed road centrelines; no label or valid-fraction selection",\n        "tiles": tiles, "blocks": sorted(blocks.values(), key=lambda item: (item["block_row"], item["block_col"]))}\n\n\ndef download_inference_blocks(stack, plan, cache_dir, *, max_workers=4, timeout=180,\n                               retries=3, data_signature=None):\n    """Download nine-band feature+valid stacks; fail explicitly on any failed block."""\n    if not 1 <= max_workers <= 4:\n        raise ValueError("Use one to four concurrent Earth Engine downloads.")\n    cache_dir = Path(cache_dir)\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    serialized = stack.serialize()\n    identity = {"stack_sha256": hashlib.sha256(serialized.encode()).hexdigest(),\n        "data_signature": data_signature, "features": FEATURE_NAMES, "extra_band": "valid",\n        "crs": METRIC_CRS, "origin": [GRID_XMIN, GRID_YTOP],\n        "scale": SCALE, "dimensions": [BLOCK_SIZE, BLOCK_SIZE]}\n    cache_key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:16]\n    directory = cache_dir / cache_key\n    directory.mkdir(exist_ok=True)\n    _json_atomic(directory / "cache_config.json", identity)\n    _json_atomic(directory / "inference_plan.json", plan)\n\n    def fetch(block):\n        path = directory / (block["block_id"] + ".tif")\n        meta_path = path.with_suffix(".json")\n        if path.exists() and meta_path.exists():\n            meta = json.loads(meta_path.read_text())\n            if meta.get("sha256") != _file_hash(path):\n                raise ValueError(f"Cached block hash mismatch: {path.name}")\n        else:\n            error = None\n            for attempt in range(retries):\n                try:\n                    url = stack.getDownloadURL({"crs": METRIC_CRS,\n                        "crs_transform": block["transform"],\n                        "dimensions": [BLOCK_SIZE, BLOCK_SIZE],\n                        "format": "GEO_TIFF", "filePerBand": False})\n                    response = requests.get(url, timeout=timeout)\n                    response.raise_for_status()\n                    temporary = path.with_suffix(".tif.tmp")\n                    temporary.write_bytes(response.content)\n                    with rasterio.open(temporary) as dataset:\n                        if dataset.count != 9 or dataset.width != BLOCK_SIZE or dataset.height != BLOCK_SIZE:\n                            raise ValueError("Expected a nine-band 512×512 input block.")\n                        if dataset.crs.to_epsg() != 2193 or not dataset.transform.almost_equals(Affine(*block["transform"])):\n                            raise ValueError("Downloaded block CRS/grid mismatch.")\n                    temporary.replace(path)\n                    meta = {**block, "sha256": _file_hash(path), "path": str(path), "cache_key": cache_key}\n                    _json_atomic(meta_path, meta)\n                    error = None\n                    break\n                except Exception as exception:\n                    error = exception\n                    if attempt + 1 < retries:\n                        time.sleep(2 * (attempt + 1))\n            if error is not None:\n                raise RuntimeError(f"{block[\'block_id\']} failed after {retries} attempts: {error}") from error\n        with rasterio.open(path) as dataset:\n            if dataset.count != 9 or dataset.width != BLOCK_SIZE or dataset.height != BLOCK_SIZE:\n                raise ValueError(f"Cached block shape mismatch: {path.name}")\n            if dataset.crs.to_epsg() != 2193 or not dataset.transform.almost_equals(Affine(*block["transform"])):\n                raise ValueError(f"Cached block CRS/grid mismatch: {path.name}")\n        return {**block, "path": str(path), "sha256": _file_hash(path), "cache_key": cache_key}\n\n    records, errors = [], []\n    with ThreadPoolExecutor(max_workers=max_workers) as executor:\n        futures = {executor.submit(fetch, block): block for block in plan["blocks"]}\n        for future in as_completed(futures):\n            block = futures[future]\n            try:\n                records.append(future.result())\n                print(f"Input block ready {len(records)}/{len(plan[\'blocks\'])}: {block[\'block_id\']}", flush=True)\n            except Exception as exception:\n                errors.append({"block_id": block["block_id"], "error": str(exception)})\n    _json_atomic(directory / "download_errors.json", errors)\n    if errors:\n        raise RuntimeError(f"{len(errors)} inference blocks failed; see {directory/\'download_errors.json\'}. No unaffected-road result was produced.")\n    records.sort(key=lambda item: (item["block_row"], item["block_col"]))\n    _json_atomic(directory / "downloaded_blocks.json", records)\n    return records\n\n\ndef infer_road_tiles(model, plan, block_records, out_dir, *, threshold=DEFAULT_THRESHOLD,\n                     device=None, batch_size=16, zero_invalid_features=False,\n                     checkpoint_metadata=None):\n    """Run the frozen model only on selected original 128×128 windows in Colab.\n\n    Probability, binary mask, and polygon outputs exclude invalid pixels. Default\n    input handling reproduces the existing benchmark normalize_features exactly;\n    zero_invalid_features can be enabled only if used by checkpoint training.\n    """\n    import torch\n    try:\n        from .landslide_benchmark import normalize_features\n    except ImportError:\n        from landslide_benchmark import normalize_features\n    if not 1 <= batch_size <= 16 or not 0 < threshold < 1:\n        raise ValueError("Use batch_size 1–16 and a probability threshold between zero and one.")\n    out_dir = Path(out_dir)\n    tile_dir = out_dir / "inference_tiles"\n    tile_dir.mkdir(parents=True, exist_ok=True)\n    blocks = {item["block_id"]: item for item in block_records}\n    missing = sorted({item["block_id"] for item in plan["tiles"]} - blocks.keys())\n    if missing:\n        raise RuntimeError(f"Missing input blocks {missing}; inference cannot mark roads unaffected.")\n    device = torch.device(device) if device is not None else next(model.parameters()).device\n    model = model.to(device)\n    model.eval()\n    roi = aoi_metric()\n    hazard_rows, coverage_rows, tile_statistics = [], [], []\n    tiles = plan["tiles"]\n    with torch.inference_mode():\n        for start in range(0, len(tiles), batch_size):\n            batch = tiles[start:start + batch_size]\n            features, raw_masks = [], []\n            for tile in batch:\n                window = Window(tile["local_col"] * TILE_SIZE, tile["local_row"] * TILE_SIZE,\n                                TILE_SIZE, TILE_SIZE)\n                with rasterio.open(blocks[tile["block_id"]]["path"]) as dataset:\n                    array = dataset.read(window=window)\n                    transform = dataset.window_transform(window)\n                if array.shape != (9, TILE_SIZE, TILE_SIZE) or not transform.almost_equals(Affine(*tile["transform"])):\n                    raise ValueError(f"Original tile window/grid mismatch: {tile[\'tile\']}")\n                valid = (array[8] > 0.5) & np.isfinite(array[:8]).all(axis=0)\n                x = normalize_features(array[:8])\n                if zero_invalid_features:\n                    x[:, ~valid] = 0\n                features.append(x)\n                raw_masks.append(valid)\n            inputs = torch.from_numpy(np.stack(features)).to(device)\n            probabilities = torch.sigmoid(model(inputs))[:, 0].cpu().numpy()\n            if probabilities.shape != (len(batch), TILE_SIZE, TILE_SIZE) or not np.isfinite(probabilities).all():\n                raise RuntimeError("Invalid neural probability output; no road-impact result produced.")\n            for tile, probability, valid in zip(batch, probabilities, raw_masks):\n                transform = Affine(*tile["transform"])\n                mask = (probability >= threshold).astype("uint8")\n                profile = {"driver": "GTiff", "width": TILE_SIZE, "height": TILE_SIZE,\n                    "count": 1, "crs": METRIC_CRS, "transform": transform,\n                    "compress": "deflate", "dtype": "float32", "nodata": -9999}\n                with rasterio.open(tile_dir / (tile["tile"] + "_probability.tif"), "w", **profile) as dataset:\n                    dataset.write(np.where(valid, probability, -9999).astype("float32"), 1)\n                profile.update(dtype="uint8", nodata=255)\n                with rasterio.open(tile_dir / (tile["tile"] + "_mask.tif"), "w", **profile) as dataset:\n                    dataset.write(np.where(valid, mask, 255).astype("uint8"), 1)\n                for geometry, value in shapes(valid.astype("uint8"), mask=valid, transform=transform):\n                    if value == 1:\n                        for part in polygon_parts(make_valid(shape(geometry)).intersection(roi)):\n                            coverage_rows.append({"tile": tile["tile"], "geometry": part})\n                for geometry, value in shapes(mask, mask=valid & (mask == 1), transform=transform):\n                    if value == 1:\n                        for part in polygon_parts(make_valid(shape(geometry)).intersection(roi)):\n                            hazard_rows.append({"tile": tile["tile"], "geometry": part})\n                tile_statistics.append({"tile": tile["tile"], "valid_pixels": int(valid.sum()),\n                    "positive_pixels": int((valid & (mask == 1)).sum()),\n                    "valid_fraction": float(valid.mean())})\n            print(f"Frozen U-Net inference: {min(start+batch_size,len(tiles))}/{len(tiles)} original tiles", flush=True)\n    hazards = gpd.GeoDataFrame(hazard_rows, crs=METRIC_CRS) if hazard_rows else empty_geodataframe(["tile", "geometry"])\n    coverage = gpd.GeoDataFrame(coverage_rows, crs=METRIC_CRS) if coverage_rows else empty_geodataframe(["tile", "geometry"])\n    _json_atomic(out_dir / "inference_manifest.json", tile_statistics)\n    _json_atomic(out_dir / "inference_config.json", {"threshold": float(threshold),\n        "feature_order": FEATURE_NAMES, "tile_size": TILE_SIZE, "scale": SCALE,\n        "origin": [GRID_XMIN, GRID_YTOP], "crs": METRIC_CRS,\n        "zero_invalid_features": bool(zero_invalid_features), "batch_size": batch_size,\n        "checkpoint_metadata": checkpoint_metadata,\n        "selection": plan["selection"], "training": "none; frozen inference only"})\n    return {"hazards": hazards, "coverage": coverage, "tiles": tile_statistics,\n            "threshold": float(threshold), "tile_directory": str(tile_dir)}\n\n\ndef _polygon_layer(layer, roi):\n    if layer.crs is None:\n        raise ValueError("Hazard/coverage polygon input needs an explicit CRS.")\n    layer = layer.to_crs(METRIC_CRS)\n    rows = []\n    for geometry in layer.geometry:\n        if geometry is None or geometry.is_empty:\n            continue\n        for part in polygon_parts(make_valid(geometry).intersection(roi)):\n            rows.append({"geometry": part})\n    return gpd.GeoDataFrame(rows, crs=METRIC_CRS) if rows else empty_geodataframe(["geometry"])\n\n\ndef assess_road_impacts(segments, predicted_hazards, observed_coverage):\n    """Generic polygon-to-road adapter retaining unknown road length explicitly."""\n    if segments.crs is None or not segments.road_id.is_unique:\n        raise ValueError("Road components need a CRS and unique road_id values.")\n    result = segments.to_crs(METRIC_CRS).copy()\n    roi = aoi_metric()\n    result = result.loc[result.geometry.intersects(roi)].copy()\n    result.geometry = result.geometry.map(make_valid).intersection(roi)\n    result = result.loc[result.length > 0].copy()\n    if result.empty:\n        raise ValueError("No nonzero road line components remain inside the agreed AOI.")\n    hazard_layer = _polygon_layer(predicted_hazards, roi)\n    coverage_layer = _polygon_layer(observed_coverage, roi)\n    coverage_union = union_all(coverage_layer.geometry.values)\n    # Restrict hazards to observed valid coverage even for externally supplied polygons.\n    hazard_union = union_all(hazard_layer.geometry.values).intersection(coverage_union)\n    impact_rows, affected_rows, unknown_rows, observed_rows = [], [], [], []\n    tolerance = 1e-6\n    for record in result.itertuples():\n        line = record.geometry.intersection(roi)\n        road_length = float(line.length)\n        observed_length, _, observed = lengths(line, coverage_union)\n        affected_length, longest, affected = lengths(line, hazard_union)\n        unknown = line_parts(line.difference(coverage_union))\n        unknown_length = sum(piece.length for piece in unknown)\n        if not affected_length <= observed_length + tolerance or not observed_length <= road_length + tolerance:\n            raise AssertionError(f"Length hierarchy failed for {record.road_id}.")\n        if abs(observed_length + unknown_length - road_length) > max(tolerance, road_length * 1e-9):\n            raise AssertionError(f"Observed/unknown lengths do not partition road {record.road_id}.")\n        if affected_length > tolerance:\n            status = "potentially_affected"\n        elif observed_length <= tolerance:\n            status = "not_assessed"\n        elif unknown_length > tolerance:\n            status = "partially_observed"\n        else:\n            status = "no_predicted_overlap"\n        impact_rows.append({"road_id": record.road_id, "road_length_m": road_length,\n            "observed_length_m": float(observed_length), "unknown_length_m": float(unknown_length),\n            "coverage_fraction": float(observed_length / road_length) if road_length else 0.0,\n            "affected_length_m": float(affected_length), "max_affected_run_m": float(longest),\n            "affected_fraction": float(affected_length / road_length) if road_length else 0.0,\n            "impact_status": status, "has_unknown_length": bool(unknown_length > tolerance),\n            "hazard_source": "frozen U-Net predicted mapped-landslide extent"})\n        for index, piece in enumerate(affected):\n            affected_rows.append({"road_id": record.road_id, "run_id": index,\n                                  "length_m": float(piece.length), "geometry": piece})\n        for index, piece in enumerate(unknown):\n            unknown_rows.append({"road_id": record.road_id, "run_id": index,\n                                 "length_m": float(piece.length), "geometry": piece})\n        for index, piece in enumerate(observed):\n            observed_rows.append({"road_id": record.road_id, "run_id": index,\n                                  "length_m": float(piece.length), "geometry": piece})\n    result = result.drop(columns=["road_length_m"], errors="ignore").merge(\n        pd.DataFrame(impact_rows), on="road_id", validate="one_to_one")\n    def runs(rows):\n        return gpd.GeoDataFrame(rows, crs=METRIC_CRS) if rows else empty_geodataframe(["road_id", "run_id", "length_m", "geometry"])\n    clean_hazards = gpd.GeoDataFrame({"geometry": polygon_parts(hazard_union)}, crs=METRIC_CRS)\n    clean_coverage = gpd.GeoDataFrame({"geometry": polygon_parts(coverage_union)}, crs=METRIC_CRS)\n    return {"roads": result, "affected_runs": runs(affected_rows), "unknown_runs": runs(unknown_rows),\n            "observed_runs": runs(observed_rows), "hazards": clean_hazards, "coverage": clean_coverage}\n\n\ndef _write_geojson(layer, path):\n    if layer.empty:\n        path.write_text(json.dumps({"type": "FeatureCollection", "features": []}))\n    else:\n        layer.to_crs(4326).to_file(path, driver="GeoJSON")\n\n\ndef export_integration_outputs(assessment, out_dir, *, inference_result=None,\n                                plan=None, checkpoint_metadata=None):\n    """Export complete road/coverage results and an illustrative Esri map."""\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    roads = assessment["roads"]\n    roads.drop(columns="geometry").to_csv(out_dir / "road_impact.csv", index=False)\n    roads.to_file(out_dir / "road_impact.gpkg", layer="road_segments", driver="GPKG")\n    _write_geojson(roads, out_dir / "road_impact.geojson")\n    for key in ("affected_runs", "unknown_runs", "observed_runs"):\n        layer = assessment[key]\n        _write_geojson(layer, out_dir / (key + ".geojson"))\n        # GeoPackage supports empty layers using modern GeoPandas/pyogrio.\n        layer.to_file(out_dir / (key + ".gpkg"), layer=key, driver="GPKG")\n    _write_geojson(assessment["hazards"], out_dir / "unet_landslides.geojson")\n    _write_geojson(assessment["coverage"], out_dir / "inference_coverage.geojson")\n    status_counts = roads.impact_status.value_counts().to_dict()\n    summary = {"road_parts": len(roads), "status_counts": status_counts,\n        "total_road_length_m": float(roads.road_length_m.sum()),\n        "total_observed_length_m": float(roads.observed_length_m.sum()),\n        "total_unknown_length_m": float(roads.unknown_length_m.sum()),\n        "total_affected_length_m": float(roads.affected_length_m.sum()),\n        "max_continuous_affected_run_m": float(roads.max_affected_run_m.max()) if len(roads) else 0.0,\n        "aoi_wgs84": list(AOI_WGS84), "metric_crs": METRIC_CRS,\n        "threshold": inference_result["threshold"] if inference_result else None,\n        "selected_tile_count": len(plan["tiles"]) if plan else None,\n        "download_block_count": len(plan["blocks"]) if plan else None,\n        "checkpoint_metadata": checkpoint_metadata, "training": "none; frozen model inference",\n        "limitations": ["Potential hazard overlap is not confirmed blockage or passability.",\n            "No predicted overlap never establishes an open or safe road.",\n            "Road sections without valid observations remain explicitly unknown.",\n            "OSM road snapshot date is unverified; road file is a subset, not a complete routable graph.",\n            "Esri imagery is an illustrative current mosaic with unknown acquisition dates, not verified 2016 disaster imagery.",\n            "Road IDs identify clipped line components, not junction-split routing graph edges.",\n            "No new segmentation accuracy is claimed for road-selected inference tiles."]}\n    _json_atomic(out_dir / "summary.json", summary)\n    if plan is not None:\n        _json_atomic(out_dir / "inference_plan.json", plan)\n    import folium\n    map_view = folium.Map(location=[-42.25, 173.75], zoom_start=10, tiles=None)\n    folium.TileLayer(tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",\n        attr="Esri, Maxar, Earthstar Geographics and the GIS User Community",\n        name="Esri imagery — illustrative; acquisition dates unverified", overlay=False, control=True).add_to(map_view)\n    layers = [(assessment["hazards"], "U-Net predicted landslides", "#f0a020", 1, 0.20, True),\n        (assessment["coverage"], "Valid inference coverage", "#4c9d69", 1, 0.10, False),\n        (roads, "Agreed road components", "#00c9ee", 3, 0, True),\n        (assessment["unknown_runs"], "Unknown / unobserved road sections", "#808080", 5, 0, True),\n        (assessment["affected_runs"], "Potentially affected observed runs", "#ff3b30", 5, 0, True)]\n    tooltip = folium.GeoJsonTooltip(fields=["road_id", "impact_status", "road_length_m", "observed_length_m",\n        "unknown_length_m", "affected_length_m", "max_affected_run_m"])\n    for layer, name, color, weight, opacity, show in layers:\n        if layer.empty:\n            continue\n        style = {"color": color, "weight": weight, "fillOpacity": opacity}\n        folium.GeoJson(layer.to_crs(4326).to_json(), name=name, show=show,\n            style_function=lambda feature, style=style: style,\n            tooltip=tooltip if name == "Agreed road components" else None).add_to(map_view)\n    bounds = roads.to_crs(4326).total_bounds\n    map_view.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])\n    folium.LayerControl(collapsed=False).add_to(map_view)\n    notice = "<div style=\'position:fixed;bottom:25px;left:15px;z-index:9999;background:white;padding:8px;font-size:12px\'>" \\\n        "Amber: prediction · Cyan: roads · Red: potential overlap · Gray: unknown<br>" \\\n        "Illustrative imagery; not verified 2016 imagery. No overlap ≠ safe passage.</div>"\n    map_view.get_root().html.add_child(folium.Element(notice))\n    map_view.save(out_dir / "road_impact_map.html")\n    import matplotlib.pyplot as plt\n    fig, ax = plt.subplots(figsize=(9, 9))\n    if not assessment["hazards"].empty:\n        assessment["hazards"].plot(ax=ax, color="#f0a020", alpha=0.4)\n    roads.plot(ax=ax, color="#00a5ce", linewidth=1)\n    if not assessment["unknown_runs"].empty:\n        assessment["unknown_runs"].plot(ax=ax, color="#808080", linewidth=2)\n    if not assessment["affected_runs"].empty:\n        assessment["affected_runs"].plot(ax=ax, color="#ff3b30", linewidth=2)\n    ax.set_title("Frozen U-Net predictions and potential road impacts\\nGray = unobserved; red = predicted overlap; no confirmed closure")\n    ax.set_xlabel("NZTM easting (m)")\n    ax.set_ylabel("NZTM northing (m)")\n    fig.tight_layout()\n    fig.savefig(out_dir / "road_impact_map.png", dpi=180)\n    plt.close(fig)\n    print(json.dumps(summary, indent=2), flush=True)\n    return {"summary": summary, "map": map_view, "output_directory": str(out_dir)}\n'}
module_dir = ROOT/'modules'
module_dir.mkdir(exist_ok=True)
for name, source_text in MODULE_SOURCES.items():
    (module_dir/name).write_text(source_text)
sys.path.insert(0, str(module_dir))
from unet_road_integration import (read_road_segments, plan_road_tiles,
    download_inference_blocks, infer_road_tiles, assess_road_impacts, export_integration_outputs)
print('Road-integration helpers ready.')


In [ ]:
# Identical inputs and preprocessing to the existing saved model.
from datetime import datetime, timezone

AOI = [173.5, -42.5, 174.0, -42.0]
POST_START, POST_END = '2016-11-14', '2017-02-01'
BANDS = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']
CRS, GRID = 'EPSG:2193', [10, 0, 1641080, 0, -10, 5350110]
roi = ee.Geometry.Rectangle(AOI, geodesic=False)

def cloud_mask(image):
    qa = image.select('QA60')
    clear = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return image.select(BANDS).multiply(0.0001).updateMask(clear)

collection = (ee.ImageCollection('COPERNICUS/S2_HARMONIZED').filterBounds(roi)
              .filterDate(POST_START, POST_END)
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 50)))
scene_count = collection.size().getInfo()
if scene_count == 0:
    raise RuntimeError('No imagery in the original post-event window.')
optical_projection = ee.Image(collection.first()).select('B2').projection()
post = (collection.map(cloud_mask).median().setDefaultProjection(optical_projection)
        .resample('bilinear').clip(roi))
dem_native = ee.Image('USGS/SRTMGL1_003').select('elevation')
slope_native = ee.Terrain.slope(dem_native)
terrain = (dem_native.resample('bilinear').rename('elevation')
           .addBands(slope_native.resample('bilinear').rename('slope')))
features = post.addBands(terrain).select(FEATURE_NAMES).clip(roi)
# Keep the same land mask as checkpoint training for reproducibility.
land = ee.Image('UMD/hansen/global_forest_change_2022_v1_10').select('datamask').eq(1)
valid = (features.mask().reduce(ee.Reducer.min()).And(land)
         .clip(roi).unmask(0, False).rename('valid'))
# Exactly nine bands: eight predictors + availability, without GNS labels.
stack = features.addBands(valid).unmask(0, False).toFloat()
assert stack.bandNames().getInfo() == FEATURE_NAMES + ['valid']
imagery_config = {
    'collection': 'COPERNICUS/S2_HARMONIZED',
    'post_start': POST_START, 'post_end_exclusive': POST_END,
    'scene_count': scene_count, 'scene_ids': collection.aggregate_array('system:index').getInfo(),
    'bands': FEATURE_NAMES, 'crs': CRS, 'grid': GRID,
    'aoi': AOI, 'cloud_mask': 'QA60 cloud/cirrus, scene cloud percentage <50, median',
    'terrain': 'USGS/SRTMGL1_003 elevation and slope, bilinear',
    'land_mask': 'UMD/hansen/global_forest_change_2022_v1_10 datamask ==1',
    'normalization': 'original fixed normalization; no explicit zeroing by validity mask',
}
(ROOT/'imagery_config.json').write_text(json.dumps(imagery_config, indent=2))
print('Original post-event scenes:', scene_count)
print('Pure inference inputs:', stack.bandNames().getInfo())



In [ ]:
# @title Check saved-model consistency (no edits needed)
# Reproduce one earlier validation tile using frozen weights; no model selection.
import io, requests, rasterio
from rasterio.io import MemoryFile
PROBE = {'split': 'val', 'tile': 'r031_c007', 'sha256': '506d29acd56d37410ad0e8f6c46f9ca2b0378d84955148ed2ee018ee136371fc', 'valid_pixels': 16384, 'positive_pixels': 0}
_reference_payload = '''...'''
reference_probability = np.load(io.BytesIO(base64.b64decode(_reference_payload)))['probability']
del _reference_payload
probe_row, probe_col = [int(v[1:]) for v in PROBE['tile'].split('_')]
probe_transform = [10,0,1641080+probe_col*1280,0,-10,5350110-probe_row*1280]
probe_url = stack.getDownloadURL({'crs':CRS, 'crs_transform':probe_transform,
    'dimensions':[128,128], 'format':'GEO_TIFF', 'filePerBand':False})
response = requests.get(probe_url, timeout=180)
response.raise_for_status()
with MemoryFile(response.content) as memory:
    with memory.open() as dataset:
        probe_array = dataset.read()
assert probe_array.shape == (9,128,128)
probe_valid = probe_array[8] > 0.5
assert int(probe_valid.sum()) == PROBE['valid_pixels']
with torch.inference_mode():
    probe_probability = torch.sigmoid(model(torch.from_numpy(normalize_features(probe_array[:8])[None]).to(DEVICE)))[0,0].cpu().numpy()
max_probability_difference = float(np.max(np.abs(probe_probability[probe_valid]-reference_probability)))
parity = {'tile':PROBE['tile'], 'valid_pixels':int(probe_valid.sum()),
          'max_probability_difference_from_saved_validation':max_probability_difference,
          'optical_valid_percentiles':np.percentile(probe_array[:6,probe_valid],[2,50,98]).tolist()}
assert max_probability_difference < 2e-4, 'Checkpoint/input reproducibility mismatch; stop before road inference.'
(ROOT/'parity_check.json').write_text(json.dumps(parity,indent=2))
print('Saved-model reproducibility check passed:', parity)


In [ ]:
# The current agreed road file is pinned for reproducible joins and geometry.
import requests
import geopandas as gpd

ROAD_COMMIT = '8d739079424b3b6c7077f470346a8840f20a0e1c'
ROAD_SHA256 = '127841d08d83cfc7a3ec82cf34f850d92d6d66550e650cea4304a2a1aff26b0e'
ROAD_URL = ('https://raw.githubusercontent.com/Loading-my-brain/x-band/'
            + ROAD_COMMIT + '/kaikoura_road_route/kai_route.gpkg')
road_path = ROOT/'kai_route.gpkg'
if not road_path.exists():
    response = requests.get(ROAD_URL, timeout=120)
    response.raise_for_status()
    temporary = road_path.with_suffix('.tmp')
    temporary.write_bytes(response.content)
    temporary.replace(road_path)
assert hashlib.sha256(road_path.read_bytes()).hexdigest() == ROAD_SHA256, 'Road file hash mismatch.'

roads = read_road_segments(road_path)
plan = plan_road_tiles(roads)
print(f'Agreed road components: {len(roads)} | {roads.length.sum()/1000:.2f} km')
print(f'All road-touching original tiles: {len(plan["tiles"])} | download blocks: {len(plan["blocks"])}')
print('No training/validation/test tile budget or minimum valid-fraction filter is used.')
data_signature = hashlib.sha256(json.dumps(imagery_config, sort_keys=True).encode()).hexdigest()
blocks = download_inference_blocks(stack, plan, ROOT/'input_cache', data_signature=data_signature)
print('All road input blocks ready.')



In [ ]:
# Check a 512-block window against a direct original-128 download.
from rasterio.windows import Window
first_tile = plan['tiles'][0]
first_block = next(b for b in blocks if b['block_id']==first_tile['block_id'])
with rasterio.open(first_block['path']) as dataset:
    window_array = dataset.read(window=Window(first_tile['local_col']*128,first_tile['local_row']*128,128,128))
direct_url = stack.getDownloadURL({'crs':CRS,'crs_transform':first_tile['transform'],
    'dimensions':[128,128],'format':'GEO_TIFF','filePerBand':False})
response = requests.get(direct_url,timeout=180)
response.raise_for_status()
with MemoryFile(response.content) as memory:
    with memory.open() as dataset:
        direct_array = dataset.read()
assert np.array_equal(window_array[8],direct_array[8]), 'Block/window validity masks differ.'
block_difference = float(np.max(np.abs(window_array[:8]-direct_array[:8])))
assert np.allclose(window_array,direct_array,atol=1e-5,rtol=1e-6), 'Download block changes original inputs.'
parity['block_window_tile'] = first_tile['tile']
parity['block_window_max_absolute_difference'] = block_difference
(ROOT/'parity_check.json').write_text(json.dumps(parity,indent=2))
print('512-download / 128-inference window check passed. Max input difference:',block_difference)


In [ ]:
# Frozen inference in Colab only. No retraining, threshold tuning or ground-truth input.
import shutil
import logging
from IPython.display import display, Markdown
from google.colab import files

class _TiffColorMetadataFilter(logging.Filter):
    def filter(self, record):
        return 'Sum of Photometric type-related color channels and ExtraSamples' not in record.getMessage()

logging.getLogger('rasterio._env').addFilter(_TiffColorMetadataFilter())

OUT = ROOT/'outputs'
inference = infer_road_tiles(model, plan, blocks, OUT, threshold=THRESHOLD,
                            device=DEVICE, batch_size=16,
                            checkpoint_metadata=MODEL_METADATA)
assert len(inference['tiles']) == len(plan['tiles'])
assert {t['tile'] for t in inference['tiles']} == {t['tile'] for t in plan['tiles']}
assessment = assess_road_impacts(roads, inference['hazards'], inference['coverage'])
exported = export_integration_outputs(assessment, OUT, inference_result=inference,
                                     plan=plan, checkpoint_metadata=MODEL_METADATA)
road_result = assessment['roads']
assert road_result.road_id.is_unique
assert (road_result.affected_length_m <= road_result.observed_length_m + 1e-6).all()
assert (road_result.observed_length_m <= road_result.road_length_m + 1e-6).all()
shutil.copy(ROOT/'imagery_config.json', OUT/'imagery_config.json')
shutil.copy(ROOT/'parity_check.json', OUT/'parity_check.json')
shutil.copy(checkpoint_path, OUT/checkpoint_path.name)
(OUT/'model_metadata.json').write_text(json.dumps(MODEL_METADATA, indent=2))
(OUT/'road_provenance.json').write_text(json.dumps({
    'repository': 'https://github.com/Loading-my-brain/x-band',
    'commit': ROAD_COMMIT, 'path': 'kaikoura_road_route/kai_route.gpkg',
    'sha256': ROAD_SHA256, 'attribution': 'OpenStreetMap contributors',
}, indent=2))
display(Markdown('### U-Net + roads integration result\n'
                 'Red = potential model overlap; gray = no valid observation. '
                 'Cyan does not establish safe passage. The imagery basemap is illustrative.'))
display(road_result[['road_id', 'name', 'highway', 'impact_status', 'affected_length_m',
                    'max_affected_run_m', 'coverage_fraction', 'unknown_length_m']]
        .sort_values('affected_length_m', ascending=False).head(20).round(2))
exported['map'].get_root().height = 500
display(exported['map'])
bundle = shutil.make_archive(str(ROOT/'kaikoura_unet_road_outputs'), 'zip', OUT)
print('Completed integration. Download bundle:', bundle)
files.download(bundle)


## Interpretation and handoff

- `road_impact.csv` / `.gpkg` / `.geojson`: parent `full_id`/`osm_id`, clipped `road_id`, affected/observed/unknown lengths.
- `affected_runs.geojson`: exact observed road-centreline sections inside predicted landslides; red on the map.
- `unknown_runs.geojson`: unobserved road sections, not confirmed open.
- `unet_landslides.geojson`, `inference_coverage.geojson`, and georeferenced probability/mask tiles are included.
- Join to routing with the parent IDs and agreed graph segmentation; these clipped components are not a complete routing graph.
- The 10 m output cannot establish remaining vehicle clearance, complete blockage or current passability. No new road-blockage accuracy is claimed.

Sources: [team road repository](https://github.com/Loading-my-brain/x-band), [OpenStreetMap attribution](https://www.openstreetmap.org/copyright), [Sentinel-2 catalog](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_HARMONIZED), [SRTM catalog](https://developers.google.com/earth-engine/datasets/catalog/USGS_SRTMGL1_003). The model was trained using [GNS V3 Kaikōura source/debris labels](https://doi.org/10.17603/ds2-rpyg-0h84).

In [ ]:
# Load the previously trained 150-tree RF; no training and no encoded weights in this cell.
import gzip, io, joblib, sklearn
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

RF_METADATA = {'model': 'Random Forest', 'trees': 150, 'threshold': 0.28, 'threshold_selection': 'existing validation F1 on 0.01–0.99 grid; frozen', 'training_pixels': 150000, 'seed': 761, 'features': ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'elevation', 'slope'], 'file_sha256': 'db43c611a13967c447049e6934e617a4b4111cfaf9fc8fc8671c6e0c2cd1f5ad', 'original_joblib_sha256': 'f19a1d236938093a6b905aedf80cb8893c362e356ec9495de8eef3b73c8baad6', 'training': 'previous cloud-trained baseline; no new training', 'drive_file_id': '136t-zBYg7bNVK3ye5UEI3YReoeqdpXiM'}
RF_THRESHOLD = 0.28
rf_checkpoint = ROOT/'rf_150_trees.joblib.gz'
if not rf_checkpoint.exists():
    credentials, _ = google.auth.default()
    model_drive = build('drive','v3',credentials=credentials,cache_discovery=False)
    request = model_drive.files().get_media(fileId=RF_METADATA['drive_file_id'])
    temporary = rf_checkpoint.with_suffix('.download')
    with temporary.open('wb') as output:
        transfer = MediaIoBaseDownload(output,request,chunksize=8*1024*1024)
        done = False
        while not done:
            status,done = transfer.next_chunk()
            print(f'Loading saved RF: {100*status.progress():.0f}%',flush=True)
    temporary.replace(rf_checkpoint)
assert hashlib.sha256(rf_checkpoint.read_bytes()).hexdigest()==RF_METADATA['file_sha256']
assert sklearn.__version__=='1.6.1', 'Use the pinned scikit-learn 1.6.1 setup cell, then restart runtime.'
with gzip.open(rf_checkpoint,'rb') as compressed:
    original_bytes = compressed.read()
assert hashlib.sha256(original_bytes).hexdigest()==RF_METADATA['original_joblib_sha256']
rf_model = joblib.load(io.BytesIO(original_bytes))
del original_bytes
assert rf_model.n_estimators==150 and len(rf_model.estimators_)==150
assert rf_model.n_features_in_==8 and list(rf_model.classes_)==[0,1]

# Compare a few fixed saved-validation probabilities as a reproducibility check.
rf_probe_indices = np.array([0, 528, 1057, 1585, 2114, 2642, 3171, 3699, 4228, 4756, 5285, 5813, 6342, 6870, 7399, 7927, 8456, 8984, 9513, 10041, 10570, 11098, 11627, 12155, 12684, 13212, 13741, 14269, 14798, 15326, 15855, 16383],dtype=int)
rf_probe_expected = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.004790991747656947, 0.005573168852091931, 0.0, 0.0, 0.06755642658632621, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.006165902029909369, 0.0, 0.0, 0.006051049533961453, 0.0, 0.0062099074954612265, 0.024638293955685876, 0.02891863884114688, 0.0, 0.0, 0.0, 0.0, 0.06510894697032106, 0.017732386642129453, 0.028161678835819513],dtype='float32')
rf_probe_features = normalize_features(probe_array[:8]).reshape(8,-1).T[rf_probe_indices]
rf_probe_actual = rf_model.predict_proba(rf_probe_features)[:,1]
rf_parity_error = float(np.max(np.abs(rf_probe_actual-rf_probe_expected)))
assert rf_parity_error<1e-6, f'Saved RF feature/parity mismatch: {rf_parity_error}'
print('Saved RF ready: 150 trees | frozen threshold 0.28 | parity max error',rf_parity_error)


In [ ]:
# GNS reference labels: complete inventory in the agreed AOI, not only road-touching polygons.
from concurrent.futures import ThreadPoolExecutor, as_completed
import geopandas as gpd
import pandas as pd
import json, time
import requests, rasterio
from rasterio.transform import Affine

GNS_ROOT = ROOT/'gns_reference'
GNS_ROOT.mkdir(exist_ok=True)
GNS_SOURCE_ASSET = 'projects/deep-voyage-479406-j9/assets/kaikoura_source_v3'
GNS_DEBRIS_ASSET = 'projects/deep-voyage-479406-j9/assets/kaikoura_debris_v3'
gns_source = ee.FeatureCollection(GNS_SOURCE_ASSET).filterBounds(roi)
gns_debris = ee.FeatureCollection(GNS_DEBRIS_ASSET).filterBounds(roi)
print('Checking GNS reference access:',gns_source.size().getInfo(),'source polygons,',
      gns_debris.size().getInfo(),'debris polygons',flush=True)

def download_reference_vectors(asset_id, name):
    path = GNS_ROOT/(name+'.geojson')
    if not path.exists():
        collection = ee.FeatureCollection(asset_id).filterBounds(roi).select([])
        features, token = [], None
        while True:
            params = {'expression':collection,'pageSize':1000}
            if token:
                params['pageToken'] = token
            page = ee.data.computeFeatures(params)
            features.extend(page.get('features',[]))
            token = page.get('nextPageToken')
            print(name,'reference polygons fetched:',len(features),flush=True)
            if not token:
                break
        if not features:
            raise RuntimeError('Empty GNS reference layer: '+asset_id)
        temporary = path.with_suffix('.tmp')
        temporary.write_text(json.dumps({'type':'FeatureCollection','features':features}))
        temporary.replace(path)
    return path

with ThreadPoolExecutor(max_workers=2) as executor:
    reference_paths = list(executor.map(lambda item:download_reference_vectors(*item),
        [(GNS_SOURCE_ASSET,'source'),(GNS_DEBRIS_ASSET,'debris')]))
gns_frames = [gpd.read_file(p).to_crs('EPSG:2193')[['geometry']] for p in reference_paths]
gns_polygons = gpd.GeoDataFrame(pd.concat(gns_frames,ignore_index=True),crs='EPSG:2193')

# Same GNS union target and paint grid used to train/validate the saved models.
gns_label = (ee.Image.constant(0).byte().setDefaultProjection(CRS,GRID)
             .paint(gns_source.merge(gns_debris),1).rename('gns_landslide')
             .clip(roi).unmask(0,False))
label_dir = GNS_ROOT/'label_blocks'
label_dir.mkdir(exist_ok=True)

def download_reference_block(block):
    path = label_dir/(block['block_id']+'.tif')
    if not path.exists():
        for attempt in range(3):
            try:
                url = gns_label.getDownloadURL({'crs':CRS,'crs_transform':block['transform'],
                    'dimensions':[512,512],'format':'GEO_TIFF','filePerBand':False})
                response = requests.get(url,timeout=180)
                response.raise_for_status()
                temporary = path.with_suffix('.tmp')
                temporary.write_bytes(response.content)
                with rasterio.open(temporary) as dataset:
                    assert (dataset.count,dataset.width,dataset.height)==(1,512,512)
                    assert dataset.crs.to_epsg()==2193
                    assert dataset.transform.almost_equals(Affine(*block['transform']))
                temporary.replace(path)
                break
            except Exception:
                if attempt==2:
                    raise
                time.sleep(2*(attempt+1))
    return block['block_id'],str(path)

reference_blocks = {}
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [executor.submit(download_reference_block,b) for b in plan['blocks']]
    for future in as_completed(futures):
        key,path = future.result()
        reference_blocks[key] = path
        print('GNS raster reference ready:',len(reference_blocks),'/',len(plan['blocks']),flush=True)
label_records = [dict(tile,path=reference_blocks[tile['block_id']]) for tile in plan['tiles']]
(GNS_ROOT/'label_manifest.json').write_text(json.dumps(label_records,indent=2))
print('GNS vector and aligned raster references ready for all',len(label_records),'road tiles.')



In [ ]:
# @title Load RF / GNS comparison tools (no edits needed)
# Embedded reviewed helpers; run model computations in Colab only.
comparison_source = '"""Inference-only RF / U-Net / GNS integration comparison.\n\nNo training or model prediction runs on import. RF inference is an explicit Colab\noperation. Reference scores on road-selected tiles are integration demonstrations,\nnot new independent held-out accuracy or confirmed road-closure accuracy.\n"""\nfrom pathlib import Path\nimport json\n\nimport geopandas as gpd\nimport numpy as np\nimport pandas as pd\nimport rasterio\nfrom rasterio.features import shapes\nfrom rasterio.transform import Affine\nfrom rasterio.windows import Window\nfrom shapely import make_valid, union_all\nfrom shapely.geometry import shape\nfrom sklearn.metrics import average_precision_score, roc_auc_score\n\ntry:\n    from .unet_road_integration import (\n        METRIC_CRS, TILE_SIZE, FEATURE_NAMES, aoi_metric, polygon_parts,\n        empty_geodataframe, assess_road_impacts, _json_atomic)\nexcept ImportError:\n    from unet_road_integration import (\n        METRIC_CRS, TILE_SIZE, FEATURE_NAMES, aoi_metric, polygon_parts,\n        empty_geodataframe, assess_road_impacts, _json_atomic)\n\n\ndef _write_comparison_geojson(layer, path):\n    """Write full-precision WGS84 GeoJSON with valid polygon geometry.\n\n    Repair after projection, retain polygon components for polygon inputs, and\n    avoid the OGR GeoJSON driver\'s default coordinate rounding. Validate a\n    read-back before replacing the destination file. Metrics are not recomputed.\n    """\n    path = Path(path)\n    if layer.empty:\n        path.write_text(json.dumps({"type": "FeatureCollection", "features": []}))\n        return\n    if layer.crs is None:\n        raise ValueError("GeoJSON export requires an explicit input CRS.")\n    geographic = layer.to_crs(4326).copy()\n    repaired = []\n    for original in geographic.geometry:\n        if original is None or original.is_empty:\n            repaired.append(None)\n            continue\n        fixed = make_valid(original)\n        if original.geom_type in {"Polygon", "MultiPolygon"}:\n            parts = polygon_parts(fixed)\n            fixed = union_all(parts) if parts else None\n        repaired.append(fixed)\n    geographic.geometry = repaired\n    geographic = geographic.loc[geographic.geometry.notna() & ~geographic.geometry.is_empty].copy()\n    if not geographic.geometry.is_valid.all():\n        raise ValueError(f"Invalid geometry remains after WGS84 repair: {path.name}")\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(geographic.to_json(drop_id=True))\n    if not geographic.empty:\n        roundtrip = gpd.read_file(temporary)\n        if len(roundtrip) != len(geographic) or not roundtrip.geometry.notna().all() or not roundtrip.geometry.is_valid.all():\n            raise ValueError(f"GeoJSON validity/read-back check failed: {path.name}")\n    temporary.replace(path)\n\n\ndef infer_rf_road_tiles(model, plan, block_records, out_dir, *, threshold=0.28,\n                        pixel_chunk_size=65536, expected_n_estimators=150,\n                        checkpoint_metadata=None):\n    """Apply the saved RF to valid pixels in every road-selected original tile."""\n    try:\n        from .landslide_benchmark import normalize_features\n    except ImportError:\n        from landslide_benchmark import normalize_features\n    if not 0 < threshold < 1 or not 1 <= pixel_chunk_size <= 65536:\n        raise ValueError("Use a frozen probability threshold and chunks of 1–65,536 pixels.")\n    if getattr(model, "n_features_in_", None) != 8:\n        raise ValueError("Expected the saved eight-feature Random Forest.")\n    if getattr(model, "n_estimators", None) != expected_n_estimators:\n        raise ValueError(f"Expected the frozen {expected_n_estimators}-tree RF checkpoint.")\n    estimators = getattr(model, "estimators_", None)\n    if estimators is None or len(estimators) != expected_n_estimators:\n        raise ValueError(f"Saved RF must contain exactly {expected_n_estimators} fitted estimators.")\n    classes = list(model.classes_)\n    if len(classes) != 2 or 0 not in classes or 1 not in classes:\n        raise ValueError("RF classes must be background=0 and mapped landslide=1.")\n    positive_index = classes.index(1)\n    blocks = {item["block_id"]: item for item in block_records}\n    missing = sorted({tile["block_id"] for tile in plan["tiles"]} - blocks.keys())\n    if missing:\n        raise RuntimeError(f"Missing feature blocks {missing}; RF inference cannot mark roads unaffected.")\n    out_dir = Path(out_dir)\n    tile_dir = out_dir / "inference_tiles"\n    tile_dir.mkdir(parents=True, exist_ok=True)\n    roi = aoi_metric()\n    hazard_rows, coverage_rows, manifest = [], [], []\n    for index, tile in enumerate(plan["tiles"]):\n        window = Window(tile["local_col"] * TILE_SIZE, tile["local_row"] * TILE_SIZE,\n                        TILE_SIZE, TILE_SIZE)\n        with rasterio.open(blocks[tile["block_id"]]["path"]) as dataset:\n            array = dataset.read(window=window)\n            transform = dataset.window_transform(window)\n        if array.shape != (9, TILE_SIZE, TILE_SIZE) or not transform.almost_equals(Affine(*tile["transform"])):\n            raise ValueError(f"Original feature tile window/grid mismatch: {tile[\'tile\']}")\n        valid = (array[8] > 0.5) & np.isfinite(array[:8]).all(axis=0)\n        features = normalize_features(array[:8])\n        selected = features.reshape(8, -1).T[valid.ravel()]\n        valid_probability = np.empty(len(selected), dtype="float32")\n        for start in range(0, len(selected), pixel_chunk_size):\n            probability = model.predict_proba(selected[start:start + pixel_chunk_size])\n            valid_probability[start:start + len(probability)] = probability[:, positive_index]\n        if not np.isfinite(valid_probability).all() or np.any((valid_probability < 0) | (valid_probability > 1)):\n            raise RuntimeError("Saved RF returned invalid probabilities; no road-impact result produced.")\n        probability = np.full(valid.shape, -9999, dtype="float32")\n        probability[valid] = valid_probability\n        binary = np.full(valid.shape, 255, dtype="uint8")\n        binary[valid] = (valid_probability >= threshold).astype("uint8")\n        profile = {"driver": "GTiff", "width": TILE_SIZE, "height": TILE_SIZE,\n            "count": 1, "crs": METRIC_CRS, "transform": transform, "compress": "deflate",\n            "dtype": "float32", "nodata": -9999}\n        with rasterio.open(tile_dir / (tile["tile"] + "_probability.tif"), "w", **profile) as dataset:\n            dataset.write(probability, 1)\n        profile.update(dtype="uint8", nodata=255)\n        with rasterio.open(tile_dir / (tile["tile"] + "_mask.tif"), "w", **profile) as dataset:\n            dataset.write(binary, 1)\n        for geometry, value in shapes(valid.astype("uint8"), mask=valid, transform=transform):\n            if value == 1:\n                for part in polygon_parts(make_valid(shape(geometry)).intersection(roi)):\n                    coverage_rows.append({"tile": tile["tile"], "geometry": part})\n        for geometry, value in shapes(binary, mask=valid & (binary == 1), transform=transform):\n            if value == 1:\n                for part in polygon_parts(make_valid(shape(geometry)).intersection(roi)):\n                    hazard_rows.append({"tile": tile["tile"], "geometry": part})\n        manifest.append({"tile": tile["tile"], "valid_pixels": int(valid.sum()),\n            "positive_pixels": int((binary == 1).sum()), "valid_fraction": float(valid.mean())})\n        if (index + 1) % 10 == 0 or index + 1 == len(plan["tiles"]):\n            print(f"Frozen RF inference: {index + 1}/{len(plan[\'tiles\'])} original tiles", flush=True)\n    hazards = gpd.GeoDataFrame(hazard_rows, crs=METRIC_CRS) if hazard_rows else empty_geodataframe(["tile", "geometry"])\n    coverage = gpd.GeoDataFrame(coverage_rows, crs=METRIC_CRS) if coverage_rows else empty_geodataframe(["tile", "geometry"])\n    _json_atomic(out_dir / "inference_manifest.json", manifest)\n    config = {"model": "saved Random Forest", "n_estimators": int(model.n_estimators),\n        "threshold": float(threshold), "feature_order": FEATURE_NAMES,\n        "normalization": "unchanged benchmark normalize_features", "selected_tile_count": len(plan["tiles"]),\n        "training": "none; saved checkpoint inference only", "checkpoint_metadata": checkpoint_metadata}\n    _json_atomic(out_dir / "inference_config.json", config)\n    return {"hazards": hazards, "coverage": coverage, "tiles": manifest,\n            "threshold": float(threshold), "tile_directory": str(tile_dir), "config": config}\n\n\ndef _prediction_directory(value):\n    if isinstance(value, dict):\n        value = value["tile_directory"]\n    directory = Path(value)\n    return directory if directory.name == "inference_tiles" else directory / "inference_tiles"\n\n\ndef _read_prediction(tile, directory, threshold):\n    with rasterio.open(directory / (tile + "_probability.tif")) as dataset:\n        probability, transform, crs = dataset.read(1), dataset.transform, dataset.crs\n        nodata = dataset.nodata\n    with rasterio.open(directory / (tile + "_mask.tif")) as dataset:\n        mask = dataset.read(1)\n        if dataset.crs != crs or not dataset.transform.almost_equals(transform):\n            raise ValueError(f"Mask/probability grid mismatch: {tile}")\n    if probability.shape != (TILE_SIZE, TILE_SIZE) or crs.to_epsg() != 2193:\n        raise ValueError(f"Prediction is not an original NZTM 128×128 tile: {tile}")\n    valid = np.isfinite(probability) & (mask != 255)\n    if nodata is not None:\n        valid &= probability != nodata\n    if np.any(valid & ~np.isin(mask, [0, 1])) or np.any(valid & ((probability < 0) | (probability > 1))):\n        raise ValueError(f"Invalid probability/mask values: {tile}")\n    if np.any((mask[valid] == 1) != (probability[valid] >= threshold)):\n        raise ValueError(f"Saved binary mask does not match its frozen threshold: {tile}")\n    return probability, mask == 1, valid, transform, crs\n\n\ndef _read_label(record):\n    """Read a one-band 128 label tile, or a 128 window of a larger label block.\n\n    Optional band 2 is an explicit label-validity mask. A nodata value of zero\n    does not discard legitimate background labels; inference coverage masks AOI.\n    """\n    with rasterio.open(record["path"]) as dataset:\n        if dataset.width == TILE_SIZE and dataset.height == TILE_SIZE:\n            window = None\n            transform = dataset.transform\n        else:\n            if "local_row" not in record or "local_col" not in record:\n                raise ValueError("Larger reference blocks require local_row/local_col metadata.")\n            window = Window(record["local_col"] * TILE_SIZE, record["local_row"] * TILE_SIZE,\n                            TILE_SIZE, TILE_SIZE)\n            transform = dataset.window_transform(window)\n        labels = dataset.read(1, window=window)\n        valid = np.isfinite(labels) & np.isin(labels, [0, 1])\n        if dataset.count > 1:\n            valid &= dataset.read(2, window=window) > 0.5\n        crs = dataset.crs\n        allowed = [0, 1, 255, -9999]\n        if dataset.nodata is not None:\n            allowed.append(dataset.nodata)\n        if np.any(np.isfinite(labels) & ~np.isin(labels, allowed)):\n            raise ValueError(f"Reference is not a binary GNS target: {record[\'tile\']}")\n    if labels.shape != (TILE_SIZE, TILE_SIZE) or crs.to_epsg() != 2193:\n        raise ValueError("Reference label grid must be an original NZTM 128×128 tile.")\n    return labels > 0.5, valid, transform, crs\n\n\ndef _counts(truth, predicted):\n    return {"tn": int((~truth & ~predicted).sum()), "fp": int((~truth & predicted).sum()),\n            "fn": int((truth & ~predicted).sum()), "tp": int((truth & predicted).sum())}\n\n\ndef _count_metrics(counts):\n    tn, fp, fn, tp = (counts[key] for key in ["tn", "fp", "fn", "tp"])\n    return {**counts, "precision": tp / max(tp + fp, 1), "recall": tp / max(tp + fn, 1),\n        "f1": 2 * tp / max(2 * tp + fp + fn, 1), "iou": tp / max(tp + fp + fn, 1),\n        "accuracy": (tp + tn) / max(tp + tn + fp + fn, 1),\n        "balanced_accuracy": (tp / max(tp + fn, 1) + tn / max(tn + fp, 1)) / 2}\n\n\ndef score_prediction_tiles(label_records, predictions_by_model, out_dir, *,\n                            thresholds=None, evaluation_scope="road-selected integration demonstration; may overlap model training/validation"):\n    """Compare frozen predictions against aligned GNS raster labels, never tune."""\n    thresholds = thresholds or {"U-Net": 0.31, "Random Forest": 0.28}\n    if set(predictions_by_model) != set(thresholds):\n        raise ValueError("Every model needs an explicit frozen threshold.")\n    if len({record["tile"] for record in label_records}) != len(label_records):\n        raise ValueError("Duplicate reference tile IDs.")\n    directories = {name: _prediction_directory(value) for name, value in predictions_by_model.items()}\n    counts = {name: dict(tn=0, fp=0, fn=0, tp=0) for name in directories}\n    probabilities = {name: [] for name in directories}\n    truth_parts, tile_rows = [], []\n    for record in label_records:\n        truth, label_valid, transform, crs = _read_label(record)\n        items = {name: _read_prediction(record["tile"], directory, thresholds[name])\n                 for name, directory in directories.items()}\n        shared = label_valid.copy()\n        for probability, predicted, valid, model_transform, model_crs in items.values():\n            if model_crs != crs or not model_transform.almost_equals(transform):\n                raise ValueError(f"Reference/prediction grid mismatch: {record[\'tile\']}")\n            shared &= valid\n        truth_parts.append(truth[shared])\n        for name, (probability, predicted, valid, _, _) in items.items():\n            item_counts = _counts(truth[shared], predicted[shared])\n            for key in counts[name]:\n                counts[name][key] += item_counts[key]\n            probabilities[name].append(probability[shared])\n            tile_rows.append({"tile": record["tile"], "model": name,\n                "shared_valid_pixels": int(shared.sum()), "gns_positive_pixels": int((truth & shared).sum()),\n                "threshold": float(thresholds[name]), **_count_metrics(item_counts)})\n    truth = np.concatenate(truth_parts) if truth_parts else np.array([], dtype=bool)\n    if not len(truth):\n        raise ValueError("No shared valid reference/prediction pixels are available.")\n    metrics = {}\n    for name in directories:\n        values = np.concatenate(probabilities[name])\n        result = _count_metrics(counts[name])\n        result.update(threshold=float(thresholds[name]), valid_pixels=int(len(truth)),\n                      positive_fraction=float(truth.mean()))\n        if len(np.unique(truth)) == 2:\n            result["average_precision"] = float(average_precision_score(truth, values))\n            result["roc_auc"] = float(roc_auc_score(truth, values))\n        metrics[name] = result\n    metadata = {"evaluation_scope": evaluation_scope, "threshold_tuning": "none; frozen model thresholds",\n        "reference": "GNS V3 source/debris union at original 10 m grid; not road-closure truth",\n        "tile_count": len(label_records), "shared_valid_pixels": int(len(truth)),\n        "new_held_out_accuracy_claim": False}\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    _json_atomic(out_dir / "pixel_reference_metrics.json", {"metadata": metadata, "models": metrics})\n    pd.DataFrame(metrics).T.to_csv(out_dir / "pixel_reference_metrics.csv")\n    pd.DataFrame(tile_rows).to_csv(out_dir / "pixel_reference_metrics_by_tile.csv", index=False)\n    return {"metadata": metadata, "models": metrics, "per_tile": tile_rows}\n\n\ndef compare_road_models(roads, unet_result, rf_result, gns_polygons, out_dir):\n    """Compare three hazards on exactly the same observed road-centreline portions."""\n    roi = aoi_metric()\n    unet_coverage = unet_result["coverage"].to_crs(METRIC_CRS)\n    rf_coverage = rf_result["coverage"].to_crs(METRIC_CRS)\n    shared_geometry = union_all(unet_coverage.geometry.values).intersection(\n        union_all(rf_coverage.geometry.values)).intersection(roi)\n    shared = gpd.GeoDataFrame({"geometry": polygon_parts(shared_geometry)}, crs=METRIC_CRS)\n    assessments = {\n        "U-Net": assess_road_impacts(roads, unet_result["hazards"], shared),\n        "Random Forest": assess_road_impacts(roads, rf_result["hazards"], shared),\n        "GNS reference": assess_road_impacts(roads, gns_polygons, shared),\n    }\n    provenance = {\n        "U-Net": "frozen U-Net predicted mapped-landslide extent",\n        "Random Forest": "saved 150-tree Random Forest predicted mapped-landslide extent",\n        "GNS reference": "GNS V3 mapped source/debris inventory reference; not confirmed road blockage",\n    }\n    for name, assessment in assessments.items():\n        assessment["roads"]["hazard_source"] = provenance[name]\n    first = assessments["U-Net"]["roads"]\n    table = first[["road_id", "full_id", "osm_id", "road_length_m", "observed_length_m",\n                   "unknown_length_m", "coverage_fraction", "geometry"]].copy()\n    for name, prefix in [("U-Net", "unet"), ("Random Forest", "rf"), ("GNS reference", "gns")]:\n        layer = assessments[name]["roads"]\n        selected = layer[["road_id", "affected_length_m", "max_affected_run_m", "impact_status"]].rename(\n            columns={"affected_length_m": prefix + "_affected_length_m",\n                     "max_affected_run_m": prefix + "_max_affected_run_m", "impact_status": prefix + "_status"})\n        table = table.merge(selected, on="road_id", validate="one_to_one")\n    u, r = table.unet_affected_length_m > 1e-6, table.rf_affected_length_m > 1e-6\n    table["model_agreement"] = np.select([u & r, u & ~r, ~u & r],\n        ["both", "unet_only", "rf_only"], default="neither")\n    table["has_shared_observation"] = table.observed_length_m > 1e-6\n    table["has_unknown_length"] = table.unknown_length_m > 1e-6\n    observed = table.loc[table.has_shared_observation]\n    summary = {"scope": "reference overlap on shared observed centreline portions; not confirmed road blockage",\n        "new_held_out_accuracy_claim": False, "road_parts": len(table),\n        "observed_road_parts": len(observed), "not_assessed_parts": int((~table.has_shared_observation).sum()),\n        "agreement_counts_observed_parts_only": observed.model_agreement.value_counts().to_dict(),\n        "total_observed_length_m": float(table.observed_length_m.sum()),\n        "total_unknown_length_m": float(table.unknown_length_m.sum()), "models": {}}\n    reference = observed.gns_affected_length_m.to_numpy() > 1e-6\n    for name, prefix in [("U-Net", "unet"), ("Random Forest", "rf"), ("GNS reference", "gns")]:\n        length = table[prefix + "_affected_length_m"]\n        item = {"affected_road_parts": int((length > 1e-6).sum()),\n            "affected_length_m": float(length.sum()),\n            "max_continuous_affected_run_m": float(table[prefix + "_max_affected_run_m"].max())}\n        if name != "GNS reference":\n            predicted = observed[prefix + "_affected_length_m"].to_numpy() > 1e-6\n            item["observed_roadpart_reference_metrics"] = _count_metrics(_counts(reference, predicted))\n            differences = observed[prefix + "_affected_length_m"] - observed.gns_affected_length_m\n            item["affected_length_mae_per_observed_roadpart_m"] = float(differences.abs().mean()) if len(differences) else None\n            table[prefix + "_minus_gns_length_m"] = table[prefix + "_affected_length_m"] - table.gns_affected_length_m\n        summary["models"][name] = item\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    table.drop(columns="geometry").to_csv(out_dir / "road_model_agreement.csv", index=False)\n    _write_comparison_geojson(table, out_dir / "road_model_agreement.geojson")\n    table.to_file(out_dir / "road_model_agreement.gpkg", layer="road_comparison", driver="GPKG")\n    _write_comparison_geojson(shared, out_dir / "shared_observed_coverage.geojson")\n    for name, stem in [("U-Net", "unet"), ("Random Forest", "rf"), ("GNS reference", "gns_reference")]:\n        _write_comparison_geojson(assessments[name]["hazards"], out_dir / (stem + "_hazards.geojson"))\n        _write_comparison_geojson(assessments[name]["affected_runs"], out_dir / (stem + "_affected_runs.geojson"))\n    _json_atomic(out_dir / "road_reference_summary.json", summary)\n    return {"roads": table, "assessments": assessments, "shared_coverage": shared, "summary": summary}\n\n\ndef export_comparison_visuals(comparison, out_dir, *, pixel_scores=None,\n                               plan=None, block_records=None, label_records=None,\n                               predictions_by_model=None, thresholds=None):\n    """Matched maps and label-independent qualitative samples, no model calls."""\n    import folium\n    import matplotlib.pyplot as plt\n    from matplotlib.colors import ListedColormap\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    roads, assessments = comparison["roads"], comparison["assessments"]\n    colors = {"U-Net": "#ed4337", "Random Forest": "#8c43d1", "GNS reference": "#31a354"}\n    map_view = folium.Map(location=[-42.25, 173.75], zoom_start=10, tiles=None)\n    folium.TileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",\n        attr="Esri, Maxar, Earthstar Geographics and GIS User Community",\n        name="Illustrative imagery — dates unverified; not confirmed 2016", overlay=False).add_to(map_view)\n    folium.GeoJson(roads.to_crs(4326).to_json(), name="Shared road components",\n        style_function=lambda feature: {"color": "#65c7d0", "weight": 2},\n        tooltip=folium.GeoJsonTooltip(fields=["road_id", "model_agreement", "observed_length_m", "unknown_length_m",\n            "unet_affected_length_m", "rf_affected_length_m", "gns_affected_length_m"])).add_to(map_view)\n    unknown = assessments["U-Net"]["unknown_runs"]\n    if not unknown.empty:\n        folium.GeoJson(unknown.to_crs(4326).to_json(), name="Unknown / unobserved sections",\n            style_function=lambda feature: {"color": "#808080", "weight": 4}).add_to(map_view)\n    for name in ["U-Net", "Random Forest", "GNS reference"]:\n        group = folium.FeatureGroup(name=name + " hazards + affected runs", show=name == "U-Net")\n        for key, weight, opacity in [("hazards", 1, 0.18), ("affected_runs", 5, 0)]:\n            layer = assessments[name][key]\n            if not layer.empty:\n                style = {"color": colors[name], "weight": weight, "fillOpacity": opacity}\n                folium.GeoJson(layer.to_crs(4326).to_json(), style_function=lambda feature, style=style: style).add_to(group)\n        group.add_to(map_view)\n    bounds = roads.to_crs(4326).total_bounds\n    map_view.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])\n    folium.LayerControl(collapsed=False).add_to(map_view)\n    notice = "<div style=\'position:fixed;bottom:25px;left:15px;z-index:9999;background:white;padding:8px;font-size:12px\'>" \\\n        "Toggle model/reference layers: red U-Net · purple RF · green GNS<br>" \\\n        "Gray: unknown. GNS overlap is not confirmed closure. Imagery dates unverified.</div>"\n    map_view.get_root().html.add_child(folium.Element(notice))\n    map_view.save(out_dir / "three_way_comparison.html")\n    metric_bounds = roads.total_bounds\n    padding = max(metric_bounds[2] - metric_bounds[0], metric_bounds[3] - metric_bounds[1]) * 0.02\n    fig, axes = plt.subplots(1, 3, figsize=(15, 7), sharex=True, sharey=True)\n    for ax, name in zip(axes, ["U-Net", "Random Forest", "GNS reference"]):\n        layer = assessments[name]\n        if not layer["hazards"].empty:\n            layer["hazards"].plot(ax=ax, color=colors[name], alpha=0.25)\n        roads.plot(ax=ax, color="#84bdc3", linewidth=0.8)\n        if not unknown.empty:\n            unknown.plot(ax=ax, color="#808080", linewidth=1.5)\n        if not layer["affected_runs"].empty:\n            layer["affected_runs"].plot(ax=ax, color=colors[name], linewidth=2)\n        ax.set_xlim(metric_bounds[0] - padding, metric_bounds[2] + padding)\n        ax.set_ylim(metric_bounds[1] - padding, metric_bounds[3] + padding)\n        ax.set_title(name)\n        ax.set_xlabel("NZTM easting (m)")\n    axes[0].set_ylabel("NZTM northing (m)")\n    fig.suptitle("Same roads and observed coverage: model predictions versus GNS mapped reference\\nIntegration demonstration; no confirmed road closure or new held-out accuracy")\n    fig.tight_layout()\n    fig.savefig(out_dir / "three_way_maps.png", dpi=180)\n    plt.close(fig)\n    fig, axes = plt.subplots(1, 2 if pixel_scores else 1, figsize=(12 if pixel_scores else 7, 4), squeeze=False)\n    offset = 0\n    if pixel_scores:\n        score_table = pd.DataFrame(pixel_scores["models"]).T[["f1", "iou"]]\n        score_table.plot.bar(ax=axes[0, 0], color=["#327cb8", "#f2a535"], rot=0)\n        axes[0, 0].set_ylim(0, 1)\n        axes[0, 0].set_ylabel("Pixel score against GNS")\n        axes[0, 0].set_title("Road-selected integration pixels; not held-out")\n        offset = 1\n    names = ["U-Net", "Random Forest", "GNS reference"]\n    values = [comparison["summary"]["models"][name]["affected_length_m"] / 1000 for name in names]\n    axes[0, offset].bar(names, values, color=[colors[name] for name in names])\n    axes[0, offset].set_ylabel("Observed road-centreline overlap (km)")\n    axes[0, offset].set_title("Potential impact length; not blockage accuracy")\n    fig.tight_layout()\n    fig.savefig(out_dir / "reference_metrics_and_road_lengths.png", dpi=180)\n    plt.close(fig)\n    if all(value is not None for value in [plan, block_records, label_records, predictions_by_model]):\n        thresholds = thresholds or {"U-Net": 0.31, "Random Forest": 0.28}\n        directories = {name: _prediction_directory(value) for name, value in predictions_by_model.items()}\n        label_map = {record["tile"]: record for record in label_records}\n        blocks = {record["block_id"]: record for record in block_records}\n        ordered = sorted(plan["tiles"], key=lambda record: (record["row"], record["col"]))\n        indices = np.unique(np.linspace(0, len(ordered) - 1, min(6, len(ordered))).round().astype(int))\n        selected = [ordered[index] for index in indices]\n        fig, axes = plt.subplots(len(selected), 6, figsize=(18, 3 * len(selected)), squeeze=False)\n        error_cmap = ListedColormap(["#eeeeee", "#35a657", "#dc4437", "#3367d6"])\n        error_cmap.set_bad("#808080")\n        binary_cmap = plt.get_cmap("viridis").copy()\n        binary_cmap.set_bad("#808080")\n        for row, tile in enumerate(selected):\n            window = Window(tile["local_col"] * TILE_SIZE, tile["local_row"] * TILE_SIZE, TILE_SIZE, TILE_SIZE)\n            with rasterio.open(blocks[tile["block_id"]]["path"]) as dataset:\n                array = dataset.read(window=window)\n            truth, label_valid, _, _ = _read_label(label_map[tile["tile"]])\n            items = {name: _read_prediction(tile["tile"], directory, thresholds[name]) for name, directory in directories.items()}\n            shared = label_valid.copy()\n            for value in items.values():\n                shared &= value[2]\n            rgb = np.clip(array[[2, 1, 0]].transpose(1, 2, 0) / 0.30, 0, 1)\n            rgb[~shared] = 0.5\n            images = [rgb, np.ma.masked_where(~shared, truth)]\n            for name in ["U-Net", "Random Forest"]:\n                images.append(np.ma.masked_where(~shared, items[name][1]))\n            for name in ["U-Net", "Random Forest"]:\n                predicted = items[name][1]\n                error = np.zeros(truth.shape, dtype="uint8")\n                error[truth & predicted] = 1\n                error[~truth & predicted] = 2\n                error[truth & ~predicted] = 3\n                images.append(np.ma.masked_where(~shared, error))\n            titles = ["Post-event RGB", "GNS reference", "U-Net mask", "RF mask", "U-Net errors", "RF errors"]\n            for col, (image, title) in enumerate(zip(images, titles)):\n                if col == 0:\n                    axes[row, col].imshow(image)\n                else:\n                    axes[row, col].imshow(image, cmap=error_cmap if col >= 4 else binary_cmap,\n                                         vmin=0, vmax=3 if col >= 4 else 1, interpolation="nearest")\n                axes[row, col].axis("off")\n                if row == 0:\n                    axes[row, col].set_title(title, pad=24)\n            axes[row, 0].text(0, 1.03, tile["tile"], transform=axes[row, 0].transAxes)\n        fig.suptitle("Fixed road-tile quantiles; no selection by model scores\\nMasks: yellow landslide · purple background · gray invalid\\nErrors against GNS: green TP · red FP · blue FN")\n        fig.tight_layout(rect=[0, 0, 1, 0.97])\n        fig.savefig(out_dir / "deterministic_tile_examples.png", dpi=160)\n        plt.close(fig)\n        _json_atomic(out_dir / "qualitative_example_selection.json", {"rule": "even index quantiles of road-selected tiles ordered north–south, then west–east; no metric selection",\n            "tiles": [tile["tile"] for tile in selected]})\n    return {"map": map_view, "output_directory": str(out_dir)}\n'
(module_dir/'road_model_comparison.py').write_text(comparison_source)
import importlib
importlib.invalidate_caches()
if "road_model_comparison" in sys.modules:
    importlib.reload(sys.modules["road_model_comparison"])
from road_model_comparison import (infer_rf_road_tiles, score_prediction_tiles,
    compare_road_models, export_comparison_visuals)
print('RF / U-Net / GNS comparison tools ready.')


In [ ]:
# @title Run U-Net / RF / GNS comparison and show results
# Compare frozen predictions on exactly the same road-selected imagery and reference grid.
import pandas as pd
from IPython.display import Image, Markdown, display

RF_OUT = ROOT/'rf_outputs'
COMPARISON_OUT = ROOT/'comparison_outputs'
# Reuse this exact in-memory result when reformatting the comparison figures.
if 'rf_result' not in globals():
    rf_result = infer_rf_road_tiles(rf_model, plan, blocks, RF_OUT,
        threshold=RF_THRESHOLD, checkpoint_metadata=RF_METADATA)
else:
    assert rf_result['threshold']==RF_THRESHOLD
    assert rf_result['config']['checkpoint_metadata']==RF_METADATA
    assert {t['tile'] for t in rf_result['tiles']}=={t['tile'] for t in plan['tiles']}
    print('Reusing unchanged frozen RF predictions for this comparison.')
pixel_scores = score_prediction_tiles(label_records,
    {'U-Net':inference, 'Random Forest':rf_result}, COMPARISON_OUT,
    thresholds={'U-Net':THRESHOLD,'Random Forest':RF_THRESHOLD},
    evaluation_scope='Road-selected integration pixels; may overlap training/validation; not independent test accuracy')
comparison = compare_road_models(roads, inference, rf_result, gns_polygons, COMPARISON_OUT)
visuals = export_comparison_visuals(comparison, COMPARISON_OUT,
    pixel_scores=pixel_scores, plan=plan, block_records=blocks,
    label_records=label_records, predictions_by_model={'U-Net':inference,'Random Forest':rf_result})

display(Markdown('## U-Net, Random Forest and GNS reference comparison\n'
    'Both saved models use the same post-event imagery and terrain features. '
    '**GNS is the mapped landslide reference, not confirmed road blockage truth.** '
    'These road-selected tiles may overlap training/validation, so the following '
    'scores describe integration agreement and are not a new independent test.'))
pixel_table = pd.DataFrame(pixel_scores['models']).T
display(pixel_table[['precision','recall','f1','iou','average_precision','accuracy']].round(4))
print('Shared valid pixels:',pixel_scores['metadata']['shared_valid_pixels'])
length_table = pd.DataFrame(comparison['summary']['models']).T
length_table['affected_length_km'] = length_table.affected_length_m.astype(float)/1000
display(length_table[['affected_road_parts','affected_length_km','max_continuous_affected_run_m']].round(3))
print('Shared observed road length (km):',comparison['summary']['total_observed_length_m']/1000)
print('Unknown road length (km):',comparison['summary']['total_unknown_length_m']/1000)
print('A larger overlap length is not evidence of a better blockage detector.')

display(Image(filename=str(COMPARISON_OUT/'reference_metrics_and_road_lengths.png')))
display(Image(filename=str(COMPARISON_OUT/'deterministic_tile_examples.png')))
display(Image(filename=str(COMPARISON_OUT/'three_way_maps.png')))
visuals['map'].get_root().height = 550
display(visuals['map'])

(COMPARISON_OUT/'rf_model_metadata.json').write_text(json.dumps(RF_METADATA,indent=2))
(COMPARISON_OUT/'comparison_provenance.json').write_text(json.dumps({
    'gns_source_asset':GNS_SOURCE_ASSET, 'gns_debris_asset':GNS_DEBRIS_ASSET,
    'gns_citation':'Jones et al., GNS V3 Kaikoura landslide inventory, DOI:10.17603/ds2-rpyg-0h84',
    'gns_use':'Mapped source/debris union; not confirmed road blockage labels',
    'raster_grid':GRID,'crs':CRS,'unet_threshold':THRESHOLD,'rf_threshold':RF_THRESHOLD,
    'road_commit':ROAD_COMMIT,'road_sha256':ROAD_SHA256,
    'rf_parity_max_probability_error':rf_parity_error,
    'evaluation_scope':'Road-selected integration demonstration, not independent held-out accuracy',
},indent=2))
shutil.copy(ROOT/'imagery_config.json',COMPARISON_OUT/'imagery_config.json')
shutil.copy(ROOT/'parity_check.json',COMPARISON_OUT/'unet_parity_check.json')
shutil.copy(RF_OUT/'inference_config.json',COMPARISON_OUT/'rf_inference_config.json')
shutil.copy(RF_OUT/'inference_manifest.json',COMPARISON_OUT/'rf_inference_manifest.json')
comparison_bundle = shutil.make_archive(str(ROOT/'kaikoura_rf_unet_gns_comparison'),'zip',COMPARISON_OUT)
print('Comparison finished. Tables, maps, reference overlaps and metrics:',comparison_bundle)
files.download(comparison_bundle)
